# NovaBank Digital Feature Engineering

Build reusable digital-banking feature families from tiny NovaBank Digital data.

This v0.4 module mirrors the v0.3 private-banking feature-engineering shape for
the digital track. It consumes the package `db_` feature library and the digital
**Progressive data views** instead of reimplementing feature logic inline, then
maps every feature family to an existing digital Detection pattern identifier.

**Glossary reminder:** a **Client** is the legal customer, a **User** is the
digital login identity, and a **Banking relationship** is the relationship
container. NovaBank Digital is a fictional institution.

In [ ]:
from pathlib import Path

import pandas as pd

from banking_fraud_lab import (
    build_digital_banking_features,
    build_foundation_progressive_views,
    build_learner_facing_views,
    generate_digital_fraud_scenarios_world,
    load_tables_to_sqlite,
)
from banking_fraud_lab.features import DIGITAL_BANKING_FEATURE_FAMILIES
from banking_fraud_lab.progressive_views import NB_USER_SESSION_CONTEXT
from banking_fraud_lab.schema import PATTERN_IDS, PROTECTED_SCENARIO_ANSWER_KEYS

pd.set_option("display.max_columns", 40)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


repo_root = find_repo_root(Path.cwd())

## Generate Learner-Facing Data

The canonical generator retains protected scenario answer keys for internal
validation. The learner-facing views drop them so this notebook never exposes a
clean ground-truth label. We generate the expanded v0.4 digital scenario mix so
the feature families have realistic signals to explore.

In [ ]:
tables = generate_digital_fraud_scenarios_world(
    seed=42,
    scale="tiny",
    scenario_prevalence=0.5,
    noisy_outcome_rate=0.3,
)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS in tables
assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables
list(learner_tables)

## Explore The Digital Session Context View

The `nb_user_session_context` Progressive data view joins the digital Client,
User, and session telemetry chain in one traceable surface. It keeps Client and
User identities explicit and carries device, authentication, network, and
geolocation context for session and payment-fraud exercises.

In [ ]:
session_context = build_foundation_progressive_views(learner_tables)[
    NB_USER_SESSION_CONTEXT.name
]
assert list(session_context.columns) == list(NB_USER_SESSION_CONTEXT.columns)
assert set(session_context["institution_name"]) == {"NovaBank Digital"}
session_context[
    [
        "session_id",
        "user_id",
        "channel",
        "device_fingerprint_hash",
        "ip_country",
        "asn_risk_score",
        "is_vpn_or_proxy",
        "auth_method",
        "session_event",
    ]
].head(8)

## Map Feature Families To Detection Patterns

The feature-family registry links every output column to a Detection pattern
identifier from `PATTERN_IDS`. The v0.4 digital families reuse the three
existing digital pattern IDs only — no new pattern IDs are introduced.

In [ ]:
feature_family_map = pd.DataFrame(
    [
        {
            "family_id": spec.family_id,
            "display_name": spec.display_name,
            "detection_pattern_id": spec.detection_pattern_id,
            "output_columns": ", ".join(spec.output_columns),
        }
        for spec in DIGITAL_BANKING_FEATURE_FAMILIES
    ]
)

assert set(feature_family_map["detection_pattern_id"]).issubset(set(PATTERN_IDS))
assert set(feature_family_map["detection_pattern_id"]) <= {
    "digital_scam_to_mule",
    "new_beneficiary_payment",
    "session_payment_velocity",
}
feature_family_map

## Build The Python Feature Frame

`build_digital_banking_features()` calculates the complete NovaBank Digital
transaction-level feature frame from the `db_` library. It is scoped to NovaBank
Digital transactions and excludes `protected_scenario_answer_keys`.

In [ ]:
feature_frame = build_digital_banking_features(learner_tables)
expected_feature_columns = {
    output_column
    for spec in DIGITAL_BANKING_FEATURE_FAMILIES
    for output_column in spec.output_columns
}

assert expected_feature_columns <= set(feature_frame.columns)
assert PROTECTED_SCENARIO_ANSWER_KEYS not in feature_frame.columns
assert not feature_frame.empty

feature_columns_to_review = [
    "transaction_id",
    "account_id",
    "channel",
    "direction",
    "amount_chf",
    "db_is_vpn_or_proxy",
    "db_is_high_risk_network",
    "db_is_new_beneficiary",
    "db_is_early_life_account",
    "db_is_rapid_pass_through",
    "db_is_shared_device",
    "db_is_beneficiary_country_risky",
    "db_session_payment_count",
]
feature_frame.loc[:, feature_columns_to_review].head(8)

## Compare Digital And Private-Banking Patterns

Digital Detection patterns describe session, beneficiary, and pass-through
behavior, while the private-banking patterns describe relationship and
counterparty movement. Both tracks reuse the same Detection pattern registry, so
learners can compare the two without a parallel metadata system.

In [ ]:
pattern_summary = pd.DataFrame(
    [
        {
            "track": pattern.track,
            "pattern_id": pattern.pattern_id,
            "display_name": pattern.display_name,
            "institution": pattern.institution,
        }
        for pattern in __import__(
            "banking_fraud_lab.schema.detection_patterns",
            fromlist=["FOUNDATION_DETECTION_PATTERNS"],
        ).FOUNDATION_DETECTION_PATTERNS
    ]
)
pattern_summary

## Run The SQLite Feature Exercises

The same learner-facing tables can be loaded into SQLite and queried through the
representative digital SQL exercises. These exercises are learner-readable
versions of the same `db_` feature ideas.

In [ ]:
connection = load_tables_to_sqlite(learner_tables, ":memory:")

sqlite_tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",
    connection,
)
assert PROTECTED_SCENARIO_ANSWER_KEYS not in set(sqlite_tables["name"])

sql_paths = {
    "session_channel_features": repo_root / "sql/examples/09_digital_session_channel_features.sql",
    "beneficiary_passthrough_features": repo_root / "sql/examples/10_digital_beneficiary_passthrough_features.sql",
    "velocity_account_features": repo_root / "sql/examples/11_digital_velocity_account_features.sql",
}
sql_results = {
    name: pd.read_sql_query(path.read_text(encoding="utf-8"), connection)
    for name, path in sql_paths.items()
}

sql_result_summary = pd.DataFrame(
    [
        {
            "exercise": name,
            "rows": len(result),
            "columns": ", ".join(result.columns[:6]),
        }
        for name, result in sql_results.items()
    ]
)
sql_result_summary

## Compare Python And SQL Outputs

The SQL exercises reproduce the `db_` feature ideas. This smoke check compares the
Python feature frame with the SQLite outputs on the columns they share, so the
two learner paths stay aligned.

In [ ]:
python_session_channel = feature_frame.merge(
    sql_results["session_channel_features"][
        [
            "transaction_id",
            "db_is_vpn_or_proxy",
            "db_is_high_risk_network",
            "db_is_mobile_app_channel",
            "db_is_beneficiary_country_risky",
        ]
    ],
    on="transaction_id",
    suffixes=("_python", "_sql"),
    validate="one_to_one",
)
for column in (
    "db_is_vpn_or_proxy",
    "db_is_high_risk_network",
    "db_is_mobile_app_channel",
    "db_is_beneficiary_country_risky",
):
    assert (
        python_session_channel[f"{column}_python"]
        == python_session_channel[f"{column}_sql"]
    ).all()

python_velocity = feature_frame.merge(
    sql_results["velocity_account_features"][
        ["transaction_id", "db_session_payment_count", "db_is_early_life_account"]
    ],
    on="transaction_id",
    suffixes=("_python", "_sql"),
    validate="one_to_one",
)
assert (
    python_velocity["db_session_payment_count_python"]
    == python_velocity["db_session_payment_count_sql"]
).all()
assert (
    python_velocity["db_is_early_life_account_python"]
    == python_velocity["db_is_early_life_account_sql"]
).all()

feature_frame.loc[:, feature_columns_to_review].describe(include="all")

In [ ]:
connection.close()